# 03 - Cuarentena de Errores

**Objetivo:** Aislar registros con errores críticos

**Entrada:**
- Tabla: `raw.invoice_lines`
- Tabla: `raw.validacion_calidad`

**Salida:**
- Tabla: `quarantine.invoice_lines`
- Archivos: `datos/cuarentena/cuarentena_*.csv`

In [ ]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime
import sys

# Agregar el módulo de conexión al path
sys.path.insert(0, str(Path.cwd().parent))
from conexion_sqlserver import obtener_engines

print("✓ Librerías importadas")

## 1. Obtener conexión a BD

In [ ]:
engine_origen, engine_destino = obtener_engines()

print("✓ Conectado a SQL Server")

## 2. Crear tabla de cuarentena

In [ ]:
with engine_destino.connect() as conn:
    # Crear schema si no existe
    conn.execute("""
        IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = 'quarantine')
        BEGIN
            EXEC sp_executesql N'CREATE SCHEMA quarantine'
        END
    """)
    
    # Crear tabla de cuarentena
    conn.execute("""
        IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'invoice_lines' AND SCHEMA_NAME(schema_id) = 'quarantine')
        BEGIN
            CREATE TABLE quarantine.invoice_lines (
                id INT PRIMARY KEY IDENTITY(1,1),
                lote_id VARCHAR(36) NOT NULL,
                tabla_origen VARCHAR(100) NOT NULL,
                numero_fila INT NOT NULL,
                payload NVARCHAR(MAX) NOT NULL,
                tipo_error VARCHAR(50) NOT NULL,
                motivo VARCHAR(MAX),
                fecha_cuarentena DATETIME NOT NULL DEFAULT GETDATE()
            )
        END
    """)
    
    conn.commit()

print("✓ Tabla quarantine.invoice_lines creada/verificada")

## 3. Leer datos RAW y validación

In [ ]:
# Leer datos RAW
df_raw = pd.read_sql_query("SELECT * FROM raw.invoice_lines", engine_destino)

# Leer validaciones
df_validacion = pd.read_sql_query("SELECT * FROM raw.validacion_calidad", engine_destino)

print(f"✓ Leídos {len(df_raw)} registros RAW")
print(f"✓ Leídas {len(df_validacion)} validaciones")

## 4. Identificar registros con errores críticos

In [ ]:
# Definir errores críticos
ERRORES_CRITICOS = ['NULO', 'NEGATIVO']

# Obtener filas con errores críticos
df_errores = df_validacion[df_validacion['tipo_error'].isin(ERRORES_CRITICOS)]
filas_en_cuarentena = df_errores['numero_fila'].unique()

print(f"✓ Registros con errores críticos: {len(filas_en_cuarentena)}")
print(f"  Errores encontrados:")
for error_type in ERRORES_CRITICOS:
    count = len(df_errores[df_errores['tipo_error'] == error_type])
    if count > 0:
        print(f"    - {error_type}: {count}")

## 5. Preparar datos para cuarentena

In [ ]:
# Crear registros de cuarentena
registros_cuarentena = []

for fila_num in filas_en_cuarentena:
    # Obtener registro RAW
    raw_data = df_raw[df_raw['numero_fila'] == fila_num].iloc[0]
    
    # Obtener errores asociados
    errores_fila = df_errores[df_errores['numero_fila'] == fila_num]
    
    # Construir motivo
    motivos = []
    for _, error in errores_fila.iterrows():
        motivo = f"{error['tipo_error']} en {error['campo_con_error']} (valor: {error['valor_problematico']})"
        motivos.append(motivo)
    
    registros_cuarentena.append({
        'lote_id': raw_data['lote_id'],
        'tabla_origen': raw_data['tabla_origen'],
        'numero_fila': raw_data['numero_fila'],
        'payload': raw_data['payload'],
        'tipo_error': errores_fila.iloc[0]['tipo_error'],
        'motivo': ' | '.join(motivos)
    })

print(f"✓ Preparados {len(registros_cuarentena)} registros para cuarentena")

## 6. Guardar en BD

In [ ]:
if len(registros_cuarentena) > 0:
    # Convertir a DataFrame
    df_cuarentena = pd.DataFrame(registros_cuarentena)
    
    # Insertar en BD
    df_cuarentena.to_sql('invoice_lines', engine_destino, schema='quarantine', if_exists='append', index=False)
    
    print(f"✓ Insertados {len(df_cuarentena)} registros en quarantine.invoice_lines")
else:
    print("✓ No hay registros para cuarentena (datos limpios)")
    df_cuarentena = pd.DataFrame()

## 7. Guardar en archivos CSV

In [ ]:
# Crear carpeta de cuarentena
cuarentena_dir = Path.cwd() / "datos" / "cuarentena"
cuarentena_dir.mkdir(parents=True, exist_ok=True)

if len(registros_cuarentena) > 0:
    # Guardar archivo CSV
    lote_id = registros_cuarentena[0]['lote_id'][:8]
    csv_path = cuarentena_dir / f"cuarentena_LOTE_{lote_id}.csv"
    
    df_cuarentena.to_csv(csv_path, index=False, encoding='utf-8')
    
    print(f"✓ Archivo guardado: {csv_path}")
    print(f"  Tamaño: {csv_path.stat().st_size / 1024:.1f} KB")
else:
    print("✓ No hay archivos de cuarentena (datos limpios)")

## Resumen

In [ ]:
print("\n" + "="*60)
print("RESUMEN - ETAPA 03: CUARENTENA")
print("="*60)
print(f"✓ Registros procesados: {len(df_raw)}")
print(f"✓ Registros en cuarentena: {len(registros_cuarentena)}")
print(f"✓ Registros limpios: {len(df_raw) - len(registros_cuarentena)}")
print(f"✓ Tabla BD: quarantine.invoice_lines")
if len(registros_cuarentena) > 0:
    print(f"✓ Archivo CSV: {csv_path.name}")
else:
    print(f"✓ Archivo CSV: Ninguno (datos limpios)")
print("="*60)